# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My answer

My lane is primarily a **ranking / scoring task**.

The goal is to assign each pseudonymized content page a review-priority score and then rank the pages from highest to lowest priority.

This is a ranking problem because the main decision is not simply whether a page is good or bad. The main decision is which pages a content strategist or SEO reviewer should inspect first when review time is limited.

The final output will be a ranked review queue. Each row will represent one content page and will include a priority score and reason codes that explain why the page was ranked highly.

This output supports actions such as refreshing content, expanding content, improving metadata, monitoring the page, or leaving it unchanged.


In [6]:
task_type = "Ranking / Scoring"
decision = "Which content pages should a reviewer inspect first?"
output = "A ranked review queue with priority scores and reason codes"

print("Task type:", task_type)
print("Decision:", decision)
print("Output:", output)

Task type: Ranking / Scoring
Decision: Which content pages should a reviewer inspect first?
Output: A ranked review queue with priority scores and reason codes


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### My answer

For this starter-data exercise, I will use a binary decline proxy called `decline_proxy`.

The proxy will be:

- `1` when `trend_direction` is `"down"`
- `0` for all other trend-direction values

This proxy represents whether a page showed a meaningful downward direction in the observed comparison window.

It is important to call this a proxy rather than a perfect ground-truth target. The value is derived from the dataset's trend rule, so a model trained directly on it would partly learn the existing definition of decline.

In a later and more honest predictive version of the project, the preferred target would be an observed future-window outcome. For example, features could be calculated from an earlier period and the target could show whether impressions or clicks declined during a later 30-day period.

For this week's framing task, the proxy is useful for sketching what the target column would look like and for evaluating whether highly ranked pages contain more observed declining cases.

In [7]:
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "muhammetalicvs-prog/flyrank-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

df[["content_id", "trend_direction", "decline_proxy"]].head(10)

,content_id,trend_direction,decline_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### My answer

My primary success metric will be **Precision@50**.

Precision@50 measures the proportion of the first 50 recommended pages that have a positive decline proxy.

I chose this metric because the output is intended for a reviewer with limited time. The reviewer is more likely to inspect the top part of the ranked queue than every page in the dataset.

A Precision@50 value of `0.70`, for example, would mean that 35 of the first 50 recommended pages showed the observed decline proxy.

For this initial framing, I would treat a method as useful if it clearly beats a simple transparent baseline and produces a top-50 queue with a high concentration of relevant pages.

The exact acceptable threshold should not be selected only after seeing the results. The model should be compared against both the overall proxy rate and a simple fixed-rule baseline.

In [8]:
def precision_at_k(data, score_column, target_column, k=50):
    """
    Sort rows by the score column and calculate
    the positive target rate among the first k rows.
    """
    top_k = data.sort_values(
        score_column,
        ascending=False
    ).head(k)

    return top_k[target_column].mean()


overall_proxy_rate = df["decline_proxy"].mean()

print("Evaluation metric: Precision@50")
print("Overall decline-proxy rate:", round(overall_proxy_rate, 4))
print(
    "A useful ranking should outperform this reference rate "
    "and a transparent baseline."
)

Evaluation metric: Precision@50
Overall decline-proxy rate: 0.5421
A useful ranking should outperform this reference rate and a transparent baseline.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### My answer

The unit of analysis is **one pseudonymized content page**.

Each row represents one content item belonging to one pseudonymized client. The row contains content properties and aggregated performance measurements such as impressions, clicks, CTR, average position, sessions, engagement, content age, freshness, and trend direction.

The page-level unit fits the decision because the final action is also page-level. A reviewer decides whether that specific page should be refreshed, expanded, monitored, have its metadata improved, or be left unchanged.

The `content_id` and `client_id` columns are identifiers used for grouping, joining, and evaluation splits. They should not be treated as predictive features.

In [9]:
unit_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "trend_direction",
    "decline_proxy",
]

unit_df = df[unit_columns].copy()

print("Unit of analysis: one pseudonymized content page")
print("Number of rows:", len(unit_df))
print("Number of unique content pages:", unit_df["content_id"].nunique())
print("Duplicate content IDs:", unit_df["content_id"].duplicated().sum())

unit_df.head(10)

Unit of analysis: one pseudonymized content page
Number of rows: 30000
Number of unique content pages: 30000
Duplicate content IDs: 0


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,trend_direction,decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,0.76,10.6,17,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,0.05,20.3,9,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,0.09,36.5,11,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,0.49,6.2,78,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,0.13,44.0,145,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,0.03,8.5,5,0.00,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,0.00,7.0,1,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,0.06,21.2,28,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,0.09,46.0,68,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,0.16,4.9,3,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### My answer

A fixed rule is still useful as a transparent baseline, but one fixed rule is unlikely to represent the full review-priority pattern.

A page may have high impressions but weak CTR, an aging page may still perform well, and a declining page may have so little traffic that reviewing it would have limited practical value. Average position, visibility, clicks, engagement, content age, freshness, and traffic volume can interact in different ways.

For example, the rule:

`days_since_last_update >= 180 and impressions_90d >= 500`

can identify old pages that still receive visibility. However, it cannot decide how to trade off freshness risk against CTR weakness, position, engagement, traffic volume, and trend evidence across many different pages.

A scoring or machine-learning method can combine several measured signals and rank the pages by their estimated review priority.

The learned method must still be compared against a simple fixed-rule baseline. ML earns its place only if it produces a more useful top-of-queue result on held-out data.

The output remains decision-support. It does not prove why a page declined, predict Google's algorithm, or guarantee that refreshing the page will improve its performance.

In [10]:
df["fixed_rule_score"] = (
    (df["days_since_last_update"] >= 180).astype(int)
    + (df["impressions_90d"] >= 500).astype(int)
    + (df["ctr"] < df["ctr"].median()).astype(int)
)

baseline_precision_50 = precision_at_k(
    data=df,
    score_column="fixed_rule_score",
    target_column="decline_proxy",
    k=50,
)

print(
    "Fixed-rule Precision@50:",
    round(baseline_precision_50, 4)
)

df[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "fixed_rule_score",
        "decline_proxy",
    ]
].sort_values(
    "fixed_rule_score",
    ascending=False
).head(10)

Fixed-rule Precision@50: 0.64


,content_id,days_since_last_update,impressions_90d,ctr,fixed_rule_score,decline_proxy
11489,content_5feee3994adb,194,7812,0.01,3,1
3507,content_074ba6ead17b,183,533,0.00,3,1
698,content_b16bd7307b39,194,4590,0.00,3,1
11912,content_cab6d15a5215,20,2092,0.00,2,1
11911,content_8fd57923c626,22,714,0.00,2,0
11909,content_bb485e06e530,104,2080,0.00,2,1
11930,content_90f29f1e9129,22,1694,0.00,2,0
11900,content_0f1b03b06497,104,7998,0.04,2,1
28820,content_01ec3d27560e,13,1593,0.06,2,0
12008,content_b1bc831deff1,25,11744,0.03,2,1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.